# 19_final_training_data — 최종 학습데이터 만들기

**한 줄 요약:** 모델에게 줄 **최종 표**를 만든다. 한 분자당 → [식별자(SMILES)] + [fingerprint 1024개] + [WEKA가 고른 descriptor] + [정답(potency)].

**용어 미리보기**
- **fingerprint(지문)**: 분자 구조를 **0/1 비트 1024개**로 나타낸 것. 특정 부분구조가 있으면 1. (descriptor=물성 수치와 다른 표현)
- **ECFP4**: 가장 널리 쓰는 fingerprint 종류(원자 주변을 반경 2까지 본다).

**큰 흐름:** ① 준비 → ② 전체 descriptor표 + WEKA 선택목록 읽기 → ③ 각 분자의 fingerprint 계산 → ④ 네 조각을 옆으로 붙여 CSV·Excel 저장

> **📌 이 노트북 읽는 법 (처음이면 여기부터)**
> - **셀(cell)** = 코드 한 덩어리. 위에서부터 하나씩 **실행**(`Shift`+`Enter`)한다.
> - 앞 셀에서 만든 **변수**(값에 붙인 이름표)를 뒤 셀에서 계속 쓴다. 그래서 **순서대로** 실행해야 한다.
> - 코드 줄 뒤의 `# ...` 은 **주석**(설명)이며 실행에 영향이 없다.
> - 자주 나오는 것: `=`(오른쪽 값을 왼쪽 이름에 저장), `[ ]`(리스트=순서 있는 목록), `{ }`(딕셔너리=이름표-값 쌍),
>   `for x in 목록:`(목록을 하나씩 꺼내 반복), `def 함수(입력):`(재사용 작업 묶음),
>   **DataFrame(df)** = 엑셀 표처럼 행·열이 있는 데이터(판다스). `df['열이름']`으로 한 열을 고른다.

### 셀 1 — 준비 + fingerprint '생성기' 만들기
라이브러리를 불러오고, **fingerprint 생성기**를 미리 하나 만들어 둔다(`gen_ecfp`). 생성기를 한 번 만들어 재사용하면 빠르다.
`FP_BITS = 1024`는 지문 길이(비트 수)를 정하는 값.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator     # fingerprint(지문) 만드는 최신 도구
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

FP_BITS = 1024                                     # 지문 길이(비트 수)
gen_ecfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_BITS)  # ECFP4 지문 생성기

### 셀 2 — 재료 두 개 읽기
- `full`: 셀에서 만든 **전체 descriptor Excel**(분자정보 + potency + descriptor 217개).
- `sel_cols`: WEKA가 골라 남긴 CSV의 **열 이름들**(=선택된 descriptor 이름). 여기선 값이 아니라 **이름 목록만** 필요.
- `assert`: "이 조건이 참이 아니면 멈춰라". 선택된 이름이 전체 표에 다 있는지 안전 점검(오타·불일치 방지).

In [ ]:
# 입력: (1) 전체 descriptor Excel  (2) WEKA로 고른 descriptor 목록(헤더만)
FULL = 'data/HSD17B13_1to1_descriptors.xlsx'
FILT = 'data/HSD17B13_1to1_descriptors_weka_filtered.csv'

full = pd.read_excel(FULL)                                          # 전체 표(분자정보+217 descriptor)
sel_cols = [c for c in pd.read_csv(FILT, nrows=0).columns if c.lower() != 'potency']  # 선택된 descriptor 이름들(potency 제외)
print('WEKA 선택 descriptor:', len(sel_cols), '개')

miss = [c for c in sel_cols if c not in full.columns]              # 전체 표에 없는 이름이 있나 확인
assert not miss, ('full에 없는 descriptor: %s' % miss)             # 있으면 여기서 멈춤(안전장치)
print('전체 화합물:', len(full), '| potency 분포:', dict(full.potency.value_counts()))

### 셀 3 — 각 분자를 fingerprint(1024비트)로 변환
분자를 하나씩 지문(0/1 배열)으로 바꿔 `fp_rows`에 모은다. 변환 실패 분자는 건너뛰고, 성공한 행 번호만 `keep`에 남긴다.
`np.vstack`은 여러 배열을 위아래로 쌓아 표로 만든다. 열 이름은 `fp_0000`~`fp_1023`.
`base`는 지문이 성공한 분자들의 정보+descriptor(다음 셀에서 지문과 행을 맞춰 합칠 때 사용).

In [ ]:
# canonical SMILES -> ECFP4 fingerprint(1024 bit)
fp_rows, keep = [], []                               # fp_rows=지문 담을 곳, keep=성공한 행 번호
for i, smi in enumerate(full['canonical_smiles']):   # 분자를 하나씩 반복
    m = Chem.MolFromSmiles(str(smi))                 # SMILES → 분자
    if m is None:                                    # 실패하면
        continue                                     #   건너뛰기
    fp_rows.append(gen_ecfp.GetFingerprintAsNumPy(m))  # 1024개 0/1 배열로 지문 계산 후 추가
    keep.append(i)                                   # 성공한 행 번호 기록
FP = pd.DataFrame(np.vstack(fp_rows).astype(np.int8),   # 지문들을 표로 (int8=작은 정수, 용량 절약)
                  columns=['fp_%04d' % j for j in range(FP_BITS)])  # 열이름 fp_0000..fp_1023
base = full.iloc[keep].reset_index(drop=True)        # 지문 성공한 행들의 분자정보+descriptor
print('fingerprint 계산 완료:', FP.shape, '| 유효 화합물', len(base))

### 셀 4 — 네 조각을 옆으로 붙여 최종 표 만들기 → CSV·Excel 저장
`pd.concat([...], axis=1)`은 여러 표를 **좌우로(열 방향)** 이어 붙인다. 순서:
**① SMILES(식별자) → ② fingerprint 1024 → ③ WEKA 선택 descriptor → ④ potency(맨 끝)**.
정답을 마지막 열에 두는 건 학습·WEKA의 관례. 같은 표를 CSV와 Excel로 각각 저장한다.

In [ ]:
# 최종 조립: canonical_smiles + fingerprint(1024) + descriptor(선택) + potency
final = pd.concat([
    base[['canonical_smiles']].reset_index(drop=True),   # 1) 식별자(SMILES)
    FP.reset_index(drop=True),                           # 2) fingerprint 1024개
    base[sel_cols].reset_index(drop=True),               # 3) WEKA가 고른 descriptor
    base[['potency']].reset_index(drop=True),            # 4) 정답(맨 끝 열)
], axis=1)                                               # axis=1 = 좌우(열)로 붙이기
print('최종 학습데이터 shape:', final.shape,
      '(= canonical_smiles 1 + fp %d + desc %d + potency 1)' % (FP_BITS, len(sel_cols)))
print('구성 확인 → 첫 열:', final.columns[0], '| 마지막 열:', final.columns[-1])

OUT_CSV = 'data/HSD17B13_final_training_1to1.csv'
OUT_XLSX = 'data/HSD17B13_final_training_1to1.xlsx'
final.to_csv(OUT_CSV, index=False)                       # CSV로 저장
final.to_excel(OUT_XLSX, index=False)                    # Excel로 저장
print('저장 완료:')
print('  CSV :', OUT_CSV)
print('  XLSX:', OUT_XLSX)